# EEG-to-text generation: English and Persian, single-language and multilingual training

A multilingual text generator (**mBART-50**, which covers English and Persian; tuned with LoRA) learns to write the sentence a person read from that reader's word-by-word EEG. Test sentences are never seen in training.

**Inputs** (same sentence length and fixation pattern in all of them): real `eeg`, `shuffled_eeg` (EEG of random words of the same reader), `noise`, and `word_vectors` (positive control: the input contains the words, so a working pipeline must decode them).

**EEG encoders:** `continuous` (each word's EEG projected into mBART) and `vq` (an **EEG tokenizer**: each word's EEG becomes one of 512 learned discrete tokens; the codebook is shared by both languages).

**Training settings:** `en` (English only), `fa` (Persian only), `joint` (both, one shared model with a separate input layer per language).

**Evaluation:** teacher-forced (each token predicted from the true previous tokens, as in most published EEG-to-text results) and free-running generation (the model writes the whole sentence itself); the EEG model is also tested with noise and shuffled EEG; sentiment of the generated text.

Use a **GPU runtime (A100 if available)**. Each stage pushes its results to `saved_results/eeg_to_text_<stage>/`; finished runs are reused if Colab disconnects.

## Setup

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR -c user.name=colab -c user.email=colab@users.noreply.github.com pull --rebase origin $BRANCH
%cd $REPO_DIR
!pip install -q -r requirements-diagnostics.txt
!git log --oneline -1

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
FRP_DIR = f'{ARTIFACTS}/word_eeg/FRP'
TECO_TRT_DIR = f'{THESIS_ROOT}/Data/TRT_Total'
TECO_LABELS = f'{THESIS_ROOT}/Data/teco_sentiment_labels_task1.csv'
RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/followup'
LORA_RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_text_lora'
VECTOR_CACHE = '/content/vector_cache'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
os.makedirs(RESULTS, exist_ok=True)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
for path in [MAT_DIR, LABELS_CSV, WORD_EEG_DIR, TECO_TRT_DIR, TECO_LABELS]:
    print('ok ' if os.path.exists(path) else 'MISSING', path)
print('device:', DEVICE)

# GitHub push: the GITHUB_TOKEN secret (fine-grained, Contents: Read and write on your fork).
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
token = os.environ['GITHUB_TOKEN']
check = subprocess.run(['git', 'push', '--dry-run', REPO_URL.replace('https://', f'https://x-access-token:{token}@'),
                        f'HEAD:refs/heads/push-check-{os.getpid()}'], capture_output=True, text=True)
message = (check.stdout + check.stderr).replace(token, '***')
if check.returncode == 0:
    print('push check: OK, results will be pushed to GitHub')
else:
    print('push check FAILED; results will stay on Drive only. Fix the token (see the thesis notes).\n', message[-800:])

def push(stage, *sources):
    """Commit the small result files of one stage to saved_results/followup_<stage>/ and push."""
    sources = [s for s in sources if os.path.exists(s)]
    if not sources:
        print('nothing to push for', stage); return
    result = subprocess.run(['python', 'scripts/save_results_to_github.py', '--name', f'followup_{stage}', '--source',
                             *sources, '--push', '--author-name', GIT_NAME, '--author-email', GIT_EMAIL],
                            capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, '***')[-1500:])
    if result.returncode != 0:
        print(f'WARNING: push of {stage} failed; the results are on Drive in', sources)

In [ ]:
!python -m pytest -q tests/test_eeg2text.py

## Stage 1 — continuous EEG encoder (12 runs: 3 training settings × 4 inputs)

About 10 min per run on an A100 (3–4× longer on a T4). Lower `EPOCHS` for a faster first look.

In [ ]:
OUT_T = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_to_text'
EPOCHS = 15
!python scripts/run_eeg_to_text.py --zuco-word-eeg-dir "$WORD_EEG_DIR" --teco-trt-dir "$TECO_TRT_DIR" \
    --teco-labels-csv "$TECO_LABELS" --results-dir "$OUT_T" --run-tag v1 --encoders continuous \
    --settings en fa joint --inputs eeg noise shuffled_eeg word_vectors --epochs $EPOCHS --device $DEVICE
!python scripts/save_results_to_github.py --name eeg_to_text_continuous --source "$OUT_T/v1" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"


## Stage 2 — EEG tokenizer (discrete EEG tokens; 6 runs: 3 training settings × EEG and noise)

The report adds codebook use per language and how many EEG tokens English and Persian share in joint training.

In [ ]:
!python scripts/run_eeg_to_text.py --zuco-word-eeg-dir "$WORD_EEG_DIR" --teco-trt-dir "$TECO_TRT_DIR" \
    --teco-labels-csv "$TECO_LABELS" --results-dir "$OUT_T" --run-tag v1 --encoders vq \
    --settings en fa joint --inputs eeg noise --epochs $EPOCHS --device $DEVICE
!python scripts/save_results_to_github.py --name eeg_to_text_vq --source "$OUT_T/v1" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"


## Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT_T}/v1/eeg_to_text_report.md').read()))